# Load Data

加载数据，并进行一定的处理，并生成可用于后续分析的文件。

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# 输入文件路径
data_dir = Path('../../data/preprocessed')
participants_file = data_dir / 'participants.csv'
responses_file = data_dir / 'responses.csv'
clicks_file = data_dir / 'clicks.csv'

# 输出文件路径
output_dir = Path('../../data/pickle')
participants_pickle_file = output_dir / 'participants.pkl'
timeline_file = output_dir / 'timeline.pkl'
ai_events_file = output_dir / 'ai_events.pkl'
user_msgs_file = output_dir / 'user_msgs.pkl'
assistant_msgs_file = output_dir / 'assistant_msgs.pkl'
semantic_richness_file = output_dir / 'semantic_richness.pkl'
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 加载数据
participants = pd.read_csv(participants_file, encoding='utf-8-sig')
responses = pd.read_csv(responses_file, encoding='utf-8-sig')
clicks = pd.read_csv(clicks_file, encoding='utf-8-sig')

print('participants shape:', participants.shape)
print('responses shape:', responses.shape)
print('clicks shape:', clicks.shape)

# 检查缺失值
print(participants.isnull().sum())
print(responses.isnull().sum())
print(clicks.isnull().sum())

In [ ]:
# 生成被试特质主表并保存为 pickle
participant_base_cols = [
    'participant_id', 'gender', 'age', 'bfi_extroversion', 'bfi_agreeableness',
    'bfi_conscientiousness', 'bfi_neuroticism', 'bfi_openness', 'cse_total', 'ribs_total',
    'ai_attitude', 'fam_can', 'fam_balloon', 'fam_spoon', 'fam_umbrella',
    'fam_key', 'fam_rope', 'fam_brick', 'fam_knife', 'fam_hanger',
    'fam_box', 'fam_screwdriver', 'fam_sock', 'fam_sheet', 'fam_newspaper',
    'fam_tire', 'fam_shoe', 'ai_use_freq', 'ai_days_per_month', 'ai_creative_freq',
    'dat_score', 'difficulty', 'interest', 'effort', 'self_contrib',
    'ai_contrib', 'ai_accept', 'ai_harmony', 'ai_cooperate', 'use_direct',
    'use_adapt', 'use_springboard', 'use_ignore', 'reason_bottleneck', 'reason_dissatisfied',
    'reason_unique', 'reason_compare', 'reason_curiosity', 'reason_other', 'reason_not_use',
    'reason_other_text',
]

participant_base = participants.loc[:, participant_base_cols].copy()
participant_base.to_pickle(participants_pickle_file)
print('participants 形状:', participant_base.shape)
print('participants.pkl ->', participants_pickle_file)

In [ ]:
# 合并点击和AI回复：每个点击应产生一条assistant回复
# 按participant_id, item_name, trial_index分组，分别排序后按顺序匹配
def match_clicks_to_assistant(group_clicks, group_assistant):
    """将点击与assistant回复按时间顺序匹配，返回匹配后的DataFrame"""
    if len(group_clicks) == 0 or len(group_assistant) == 0:
        return pd.DataFrame()
    # 确保按时间排序
    group_clicks = group_clicks.sort_values('time').reset_index(drop=True)
    group_assistant = group_assistant.sort_values('time').reset_index(drop=True)
    # 假设点击数和回复数相等，直接合并
    # 如果不相等，按最小长度截断并警告
    min_len = min(len(group_clicks), len(group_assistant))
    if len(group_clicks) != len(group_assistant):
        print(f"警告：被试 {group_clicks['participant_id'].iloc[0]}，物品 {group_clicks['item_name'].iloc[0]}，"
              f"点击数({len(group_clicks)})与assistant回复数({len(group_assistant)})不一致，取前{min_len}条匹配")
        # FIXME: 这里由于AI回复可能由于“AI响应失败”被排除，导致时间最小的两个出现失配情况？
        group_clicks = group_clicks.iloc[:min_len]
        group_assistant = group_assistant.iloc[:min_len]
    matched = group_clicks.copy()
    matched['assistant_time'] = group_assistant['time'].values
    matched['assistant_content'] = group_assistant['content'].values
    matched['assistant_index'] = group_assistant['trial_index'].values  # 可能相同
    return matched

# 提取所有assistant回复
assistant_msgs = responses[responses['role'] == 'assistant'].copy()
user_msgs = responses[responses['role'] == 'user'].copy()

user_msgs.to_pickle(user_msgs_file)
print('user_msgs 已保存，user_msgs.pkl ->', user_msgs_file)
print('user_msgs shape:', user_msgs.shape)
assistant_msgs.to_pickle(assistant_msgs_file)
print('assistant_msgs 已保存，assistant_msgs.pkl ->', assistant_msgs_file)
print('assistant_msgs shape:', assistant_msgs.shape)

In [ ]:
# 按被试和物品分组匹配
grouped_clicks = clicks.groupby(['participant_id', 'item_name', 'trial_index'])
grouped_assistant = assistant_msgs.groupby(['participant_id', 'item_name', 'trial_index'])

matched_events_list = []
for (pid, item, trial), click_group in grouped_clicks:
    key = (pid, item, trial)
    if key in grouped_assistant.groups:
        assistant_group = grouped_assistant.get_group(key)
    else:
        assistant_group = pd.DataFrame()
    matched = match_clicks_to_assistant(click_group, assistant_group)
    if not matched.empty:
        matched_events_list.append(matched)

ai_events = pd.concat(matched_events_list, ignore_index=True)
ai_events.rename(columns={'time': 'click_time'}, inplace=True)
print('AI事件匹配完成，形状:', ai_events.shape)
# ai_events包含：participant_id, item_name, trial_index, click_index, click_time, assistant_time, assistant_content

ai_events.to_pickle(ai_events_file)
print('AI事件已保存，ai_events.pkl ->', ai_events_file)

In [ ]:
# 构建每个被试-物品的时间线事件列表
def build_timeline(pid, item, trial):
    # 用户消息
    user_cond = (user_msgs['participant_id'] == pid) & (user_msgs['item_name'] == item) & (user_msgs['trial_index'] == trial)
    user_events = user_msgs.loc[user_cond, ['time', 'content', 'role']].copy()
    user_events['type'] = 'user'
    # AI点击事件
    ai_cond = (ai_events['participant_id'] == pid) & (ai_events['item_name'] == item) & (ai_events['trial_index'] == trial)
    ai_events_sub = ai_events.loc[ai_cond, ['click_time', 'assistant_content']].copy()
    ai_events_sub.rename(columns={'click_time': 'time', 'assistant_content': 'content'}, inplace=True)
    ai_events_sub['role'] = 'assistant'
    ai_events_sub['type'] = 'ai_click'
    # 合并并按时间排序
    timeline = pd.concat([user_events, ai_events_sub], ignore_index=True, sort=False)
    timeline = timeline.sort_values('time').reset_index(drop=True)
    timeline['participant_id'] = pid
    timeline['item_name'] = item
    timeline['trial_index'] = trial
    return timeline

timeline_list = []
if not ai_events.empty:
    for (pid, item, trial) in ai_events[['participant_id', 'item_name', 'trial_index']].drop_duplicates().itertuples(index=False):
        tl = build_timeline(pid, item, trial)
        if not tl.empty:
            timeline_list.append(tl)
else:
    print('警告：ai_events 为空，跳过时间线构建')

# 合并所有时间线
if timeline_list:
    timeline_all = pd.concat(timeline_list, ignore_index=True)
    print('时间线构建完成，形状:', timeline_all.shape)
else:
    timeline_all = pd.DataFrame()
    print('警告：未构建任何时间线，timeline_all 为空')

timeline_all.to_pickle(timeline_file)
print('时间线已保存，timeline.pkl ->', timeline_file)

In [ ]:
# 物品语义丰富性

item_semantic_richness = {
    # 组1
    '气球': 93, '罐头': 83, '勺子': 73, '雨伞': 93,
    # 组2
    '钥匙': 79, '绳子': 103, '砖头': 70, '刀': 90,
    # 组3
    '衣架': 78, '盒子': 102, '螺丝刀': 72, '袜子': 81,
    # 组4
    '床单': 76, '报纸': 108, '轮胎': 69, '鞋子': 83,
}

# 保存为pickle
pd.to_pickle(item_semantic_richness, semantic_richness_file)